In [1]:
import pandas as pd

# Bereinigten Datensatz aus dem EDA-Notebook laden
df_model = pd.read_csv("data/data_processed.csv")

# Zeilen und Spalten zur Kontrolle anzeigen
print("Zeilen und Spalten:", df_model.shape)

Zeilen und Spalten: (19718, 23)


### Bereinigte Daten für die Modellierung laden

Ich lade die Datei `data/data_processed.csv`, die ich vorher im EDA-Notebook erstellt habe. Sie enthält 19.718 Zeilen und 23 Spalten. Die Datei selbst lasse ich dabei unverändert.

In [2]:
# Eingaben: alle Spalten außer der vorherzusagenden Zielspalte
X = df_model.drop(columns=["target"])

# Zielwerte: die tatsächlichen Persönlichkeitstypen
y = df_model["target"]

# Größen zur Kontrolle anzeigen
print("Eingaben:", X.shape, "Zielwerte:", y.shape)

Eingaben: (19718, 22) Zielwerte: (19718,)


### Eingaben und Zielwerte trennen

Für das Training trenne ich die Daten in Eingaben und Zielwerte. `X` enthält die 22 Eingabespalten. In `y` steht nur der zugehörige Persönlichkeitstyp aus `target`. Beide haben 19.718 Zeilen.

In [3]:
from sklearn.model_selection import train_test_split

# 20 % für den späteren Test zurückhalten, 80 % zum Trainieren verwenden
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Größen der beiden Datenteile kontrollieren
print("Training:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (15774, 22) (15774,)
Test: (3944, 22) (3944,)


### Trainings- und Testdaten aufteilen

15.774 Zeilen verwende ich für das Training. Die übrigen 3.944 Zeilen behalte ich für den abschließenden Test zurück. Mit `stratify=y` bleiben die Anteile der vier Persönlichkeitstypen in beiden Teilen ungefähr so wie im ursprünglichen Datensatz. Das ist wichtig, weil die Klassen unterschiedlich häufig vorkommen.

Die Testdaten lasse ich bis zur finalen Bewertung unberührt.

In [4]:
# Eingabespalten nach Datentyp einteilen
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

print("Zahlenspalten:", numeric_features)
print("Kategoriespalten:", categorical_features)

Zahlenspalten: ['N6', 'N8', 'N7', 'N1', 'N9', 'N10', 'N3', 'N5', 'E3', 'N2', 'E5', 'N4', 'E7', 'E4', 'age', 'C4', 'E1', 'A4', 'E10', 'E9']
Kategoriespalten: ['gender', 'hand']


### Eingabespalten nach Datentyp

Ich teile die Eingaben nach ihrem Datentyp auf. Zu den 20 Zahlenspalten gehören die 19 Fragen und `age`. `gender` und `hand` sind Kategoriespalten. Die Aufteilung brauche ich, weil ich beide Gruppen unterschiedlich vorbereite.

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Vorbereitung der Zahlenspalten festlegen
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

### Vorbereitung der Zahlenspalten

Fehlende Zahlen ersetze ich später durch den Median der jeweiligen Trainingsspalte. Danach skaliere ich die Werte. In diesem Schritt baue ich dafür nur die Pipeline. Die Berechnung passiert erst beim Training.

In [6]:
from sklearn.preprocessing import OneHotEncoder

# Vorbereitung der Kategoriespalten festlegen
categorical_pipeline = Pipeline(steps=[
    # Fehlende Angaben nicht erraten, sondern als eigene Kategorie kennzeichnen
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

categorical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",'Unknown'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto ac

### Vorbereitung der Kategoriespalten

Fehlenden Angaben in `gender` und `hand` vergebe ich später den Wert `Unknown`. Danach wende ich den `OneHotEncoder` an, um die Kategorien in Zahlenspalten umzuwandeln. Mit `handle_unknown="ignore"` kann die Pipeline auch neue Kategorien verarbeiten, die beim Training noch nicht vorkamen.

Diese Schritte passieren erst beim Training.

In [7]:
from sklearn.compose import ColumnTransformer

# Beide Vorbereitungen den passenden Eingabespalten zuordnen
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

### Zahlen- und Kategoriespalten verbinden

Im `ColumnTransformer` verbinde ich beide Vorbereitungswege: `num` für die Zahlenspalten und `cat` für die Kategoriespalten. So kann das Modell später alle Eingaben gemeinsam verarbeiten.

In [8]:
from sklearn.linear_model import LogisticRegression

# Datenvorbereitung mit einem Klassifikationsmodell verbinden
pipeline_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Vorbereitung lernen und Modell nur mit Trainingsdaten trainieren
pipeline_lr.fit(X_train, y_train)
print("Erstes Modell trainiert.")

Erstes Modell trainiert.


### Erstes Klassifikationsmodell trainieren

Als erstes Modell nutze ich eine logistische Regression. Die Datenvorbereitung und das Modell stecken gemeinsam in einer Pipeline. Mit `.fit()` berechnet die Pipeline die benötigten Werte nur aus den Trainingsdaten und trainiert danach das Modell.

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# Trainingsdaten für fünf Prüfungen aufteilen
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modell fünfmal innerhalb der Trainingsdaten trainieren und prüfen
scores_lr = cross_validate(
    pipeline_lr, X_train, y_train,
    cv=cv,
    scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"}
)

print("Accuracy je Prüfung:", scores_lr["test_accuracy"])
print("F1-Macro je Prüfung:", scores_lr["test_f1_macro"])
print("Mittlere Accuracy:", scores_lr["test_accuracy"].mean())
print("Mittlerer F1-Macro:", scores_lr["test_f1_macro"].mean())

Accuracy je Prüfung: [0.84912837 0.85451664 0.86053883 0.85641838 0.84654407]
F1-Macro je Prüfung: [0.75858773 0.77353771 0.77906612 0.77438506 0.75182814]
Mittlere Accuracy: 0.8534292579442802
Mittlerer F1-Macro: 0.7674809521662329


### Erstes Modell mit Kreuzvalidierung prüfen

Die logistische Regression erreicht in der fünfteiligen Kreuzvalidierung eine mittlere Accuracy von `0,8534` und einen F1-Macro von `0,7675`.

Beim F1-Macro zählen alle vier Persönlichkeitstypen gleich viel. Das ist hier hilfreicher als nur die Accuracy, weil die Klassen unterschiedlich häufig vorkommen. Die Testdaten verwende ich dafür noch nicht.

In [10]:
from sklearn.ensemble import RandomForestClassifier

# Zweites Modell mit derselben Datenvorbereitung festlegen
pipeline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

# Innerhalb der Trainingsdaten fünfmal trainieren und prüfen
scores_rf = cross_validate(
    pipeline_rf, X_train, y_train,
    cv=cv,
    scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"}
)

print("Mittlere Accuracy:", scores_rf["test_accuracy"].mean())
print("Mittlerer F1-Macro:", scores_rf["test_f1_macro"].mean())

Mittlere Accuracy: 0.8380246551306569
Mittlerer F1-Macro: 0.7453297912402711


### Zweites Modell prüfen und vergleichen

Als zweites Modell teste ich einen Random Forest mit derselben Datenvorbereitung und Kreuzvalidierung. Er erreicht eine mittlere Accuracy von `0,8380` und einen F1-Macro von `0,7453`.

Die logistische Regression liegt mit `0,8534` und `0,7675` bei beiden Werten vorn. Mit den bisherigen Einstellungen ist der Random Forest also nicht besser. Als Nächstes stimme ich beide Modelle ab.

In [11]:
from sklearn.model_selection import GridSearchCV

# Drei Einstellungen für die Stärke der Modellbegrenzung prüfen
search_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid={"classifier__C": [0.1, 1.0, 10.0]},
    scoring="f1_macro",
    cv=cv,
    refit=True
)

# Suche ausschließlich mit den Trainingsdaten ausführen
search_lr.fit(X_train, y_train)

print("Beste Einstellung:", search_lr.best_params_)
print("Mittlerer F1-Macro:", search_lr.best_score_)

Beste Einstellung: {'classifier__C': 10.0}
Mittlerer F1-Macro: 0.7682608592340325


### Logistische Regression abstimmen

Für den Parameter `C` teste ich die Werte `0,1`, `1,0` und `10,0` mit einer fünfteiligen Kreuzvalidierung. Den besten F1-Macro erreicht `C=10,0` mit `0,7683`.

Vor dem Tuning lag der Wert bei `0,7675`. Der Unterschied ist mit rund `0,00078` sehr klein.

In [12]:
# Drei mögliche Tiefen für die Bäume prüfen
search_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid={"classifier__max_depth": [None, 10, 20]},
    scoring="f1_macro",
    cv=cv,
    refit=True
)

# Suche nur mit Trainingsdaten ausführen
search_rf.fit(X_train, y_train)

print("Beste Einstellung:", search_rf.best_params_)
print("Mittlerer F1-Macro:", search_rf.best_score_)

Beste Einstellung: {'classifier__max_depth': None}
Mittlerer F1-Macro: 0.7453297912402711


### Random Forest abstimmen und Modelle vergleichen

Beim Random Forest teste ich für `max_depth` die Werte `None`, `10` und `20`. Am besten schneidet `None` mit einem F1-Macro von `0,7453` ab. Das ist bereits die Standardeinstellung des ersten Modells. Das Tuning bringt hier also keine Verbesserung.

Die logistische Regression liegt mit `0,7683` vorn. Deshalb wähle ich sie später für den Test mit den zurückgehaltenen Daten aus.

In [13]:

import mlflow

print("MLflow-Version:", mlflow.__version__)

MLflow-Version: 3.16.1


### MLflow vorbereiten

Ich nutze MLflow, um meine Modelle und Ergebnisse zu speichern. Installiert ist die Version 3.16.1. Dort speichere ich beide abgestimmten Modelle mit ihren Einstellungen und den Ergebnissen aus der Kreuzvalidierung.

In [14]:
from pathlib import Path
import mlflow.sklearn

tracking_db = Path("mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{tracking_db}")
mlflow.set_experiment("personality_type")

with mlflow.start_run(run_name="LogisticRegression_tuned") as run_lr:
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("C", search_lr.best_params_["classifier__C"])
    mlflow.log_metric("cv_f1_macro", search_lr.best_score_)

    model_info_lr = mlflow.sklearn.log_model(
        search_lr.best_estimator_,
        name="model",
        # Drei Beispielzeilen speichern und Zahlenspalten als float64 beschreiben
        input_example=X_train.head(3).astype(
            {column: "float64" for column in numeric_features}
        ),
        serialization_format="cloudpickle"
    )
    lr_run_id = run_lr.info.run_id

# Nur den Dateinamen ausgeben, damit kein lokaler Benutzerpfad gespeichert wird
print("Datenbank:", tracking_db.name)
print("Run-ID:", lr_run_id)
print("Modell-URI:", model_info_lr.model_uri)

2026/10/01 14:36:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/01 14:36:54 INFO mlflow.store.db.utils: Updating database tables
2026/10/01 14:36:54 INFO mlflow.tracking.fluent: Experiment with name 'personality_type' does not exist. Creating a new experiment.
2026/10/01 14:36:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Datenbank: mlflow.db
Run-ID: 8fc4677111f24798aac65c1cac66c5ab
Modell-URI: models:/m-91e46890cb024bfcb8e87c2cf88b1823


### Abgestimmte logistische Regression in MLflow speichern

Die trainierte Pipeline habe ich in MLflow gespeichert. Sie verwendet `C=10,0` und erreicht in der Kreuzvalidierung einen F1-Macro von `0,7683`. In der Pipeline ist auch die komplette Datenvorbereitung enthalten, einschließlich `Unknown` für fehlende Angaben.

Die lokale MLflow-Datenbank liegt als `mlflow.db` im Projektordner.

In [15]:
with mlflow.start_run(run_name="RandomForest_tuned") as run_rf:
    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param(
        "max_depth",
        search_rf.best_params_["classifier__max_depth"]
    )
    mlflow.log_metric("cv_f1_macro", search_rf.best_score_)

    model_info_rf = mlflow.sklearn.log_model(
        search_rf.best_estimator_,
        name="model",
        # Drei Beispielzeilen speichern und Zahlenspalten als float64 beschreiben
        input_example=X_train.head(3).astype(
            {column: "float64" for column in numeric_features}
        ),
        serialization_format="cloudpickle"
    )
    rf_run_id = run_rf.info.run_id

print("Run-ID:", rf_run_id)
print("Modell-URI:", model_info_rf.model_uri)

2026/10/01 14:36:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run-ID: 6170a4ab93874e639a0e26f750eb7d30
Modell-URI: models:/m-7951c5c989794dbe8e97bcffb008731a


### Abgestimmten Random Forest in MLflow speichern

Auch die trainierte Random-Forest-Pipeline habe ich in MLflow gespeichert. Die beste Einstellung ist `max_depth=None`. Der F1-Macro liegt bei `0,7453`.

Für den abschließenden Test wähle ich die logistische Regression, weil ihr Wert aus der Kreuzvalidierung mit `0,7683` höher ist. Die Testdaten haben diese Entscheidung nicht beeinflusst.

In [16]:
from sklearn.metrics import accuracy_score, f1_score

champion_model = search_lr.best_estimator_
y_pred_test = champion_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1_macro = f1_score(y_test, y_pred_test, average="macro")

with mlflow.start_run(run_id=lr_run_id):
    mlflow.set_tag("selection", "champion")
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_f1_macro", test_f1_macro)

print("Gewähltes Modell: Logistische Regression")
print(f"Test-Accuracy: {test_accuracy:.4f}")
print(f"Test-F1-Macro: {test_f1_macro:.4f}")
print("Modell-URI:", model_info_lr.model_uri)

Gewähltes Modell: Logistische Regression
Test-Accuracy: 0.8471
Test-F1-Macro: 0.7545
Modell-URI: models:/m-91e46890cb024bfcb8e87c2cf88b1823


### Abschließende Modellbewertung

Zum Schluss prüfe ich die ausgewählte logistische Regression einmal mit den bisher unbenutzten Testdaten. Sie erreicht eine Accuracy von `0,8471`. Das bedeutet, dass rund 84,71 Prozent der Zuordnungen richtig sind. Der F1-Macro liegt bei `0,7545`.

Der F1-Macro liegt damit rund `0,0138` unter dem Wert aus der Kreuzvalidierung. Das Ergebnis im Test ist also etwas niedriger, aber noch nah am vorherigen Wert.

Bei der Bewertung muss ich einen Punkt beachten: Laut Codebook entsteht der Zieltyp aus denselben Frageantworten, die das Modell als Eingaben nutzt. Das Modell bildet diese Zuordnung nach. Es stellt keine unabhängige psychologische Diagnose.

Das gespeicherte Modell ist unter `models:/m-7287b761a5a847db82551331398af7de` erreichbar.

In [17]:
import sys
from pathlib import Path

# Nur den Namen der aktiven Umgebung ausgeben, nicht den persönlichen Dateipfad
print(Path(sys.executable).parent.parent.name)

.venv
